# Parihaka: leave-one-strip-out (5 folds x 5 seeds) on Kaggle
Trains 25 models (about 8 minutes each on a T4), evaluates each fold's ensemble with the extended reliability metrics,
and summarises across folds. Expect about 4 to 5 hours of GPU time in total.

**Before running:** in Session options set **Accelerator** to a GPU (T4 x2 is fine) and turn **Internet** on.

**Recommended:** use *Save Version > Save & Run All (Commit)*. That runs the whole notebook in the background, so it
survives closing your browser, and the results appear in the notebook's Output tab afterwards. If you prefer to run
interactively, go cell by cell and keep the tab open. Run the quick pipeline check (step 5) first in either case.

In [ ]:
!nvidia-smi -L

## 1. Code and dependencies

In [ ]:
%cd /kaggle/working
!test -d Wildcat-Neural-Network || git clone https://github.com/kten314/Wildcat-Neural-Network.git
%cd Wildcat-Neural-Network
!git pull -q
!pip install -q zarr segmentation-models-pytorch

## 2. Data (about 1.6 GB and 6.9 MB)

In [ ]:
!mkdir -p data/raw/parihaka
!curl -fL -o data/raw/parihaka/parihaka_labels_train.npz "https://data.mendeley.com/public-files/datasets/gnvyh3msrj/files/0a0ac0af-10ab-47c3-83ac-f38fb0090d4a/file_downloaded"
!curl -fL -o data/raw/parihaka/parihaka_data_train.npz "https://data.mendeley.com/public-files/datasets/gnvyh3msrj/files/2777ae64-68e3-4cb0-aae4-588b30b7009c/file_downloaded"
!ls -lh data/raw/parihaka
!file data/raw/parihaka/*.npz

## 3. Convert to Zarr (about 8 minutes)

In [ ]:
!python src/convert_parihaka.py

## 4. Generate the five folds
Fold 3 should be reported as identical to your frozen primary split.

In [ ]:
!python src/make_folds.py

## 5. Quick pipeline check (about 5 minutes)
Trains one tiny model on fold 3 and runs the full extended evaluation. Scores mean nothing here. If anything fails, stop and send the error.

In [ ]:
!python src/train_unet.py --split configs/folds/fold3.json --epochs 1 --steps-per-epoch 10 --tag pipecheck --figs 0
!python src/ensemble_eval.py --tag pipecheck --split configs/folds/fold3.json --figs 0

## 6. Train and evaluate every fold
For each fold: five seeds, then the ensemble evaluation. Seeds with an existing checkpoint are skipped, so a re-run resumes where it stopped.

In [ ]:
import os
for k in range(5):
    split = f"configs/folds/fold{k}.json"
    for s in range(5):
        if os.path.exists(f"checkpoints/fold{k}_seed{s}.pt"):
            print(f"fold {k} seed {s}: already trained, skipping")
            continue
        print(f"===== fold {k}, seed {s} =====")
        get_ipython().system(f"python src/train_unet.py --split {split} --epochs 30 --steps-per-epoch 100 --batch 16 --tag fold{k} --seed {s} --figs 0")
    print(f"===== evaluating fold {k} =====")
    get_ipython().system(f"python src/ensemble_eval.py --tag fold{k} --split {split} --figs 0")

## 7. Summarise across folds

In [ ]:
!python src/aggregate_folds.py --prefix fold

## 8. Package the results
Creates `loso_results.zip` (JSON and figures only, a few MB). In a committed run it also appears in the Output tab.

In [ ]:
!zip -qr loso_results.zip results
!ls -lh loso_results.zip

## 9. Clean up (only after step 8 succeeded)
Deletes the data and the 25 checkpoints so the saved output stays small. Skip this in an interactive session if you still want the models.

In [ ]:
!rm -rf data checkpoints